# MCAT Study Plan

A personalized, research-backed study schedule. The core insight from the
research: **shorter, focused sessions with active recall beat long grinding
sessions.** You don't need 10 hours a day. You need 4–5 good ones.

### The evidence

| Finding | Source |
|---------|--------|
| 4–6 focused hours/day is optimal for 510+ scores; quality drops sharply beyond 6h | [Residency Advisor — survey of 510+ test takers](https://residencyadvisor.com/resources/mcat-prep/the-10-hours-a-day-mcat-study-myth-optimal-time-by-outcome) |
| 250–350 focused hours is the efficient window for 510–515 | [Residency Advisor — study hours to target score](https://residencyadvisor.com/resources/mcat-prep/study-hours-to-target-score-survey-data-from-510-test-takers) |
| Spaced repetition users were 2x more likely to pass medical entrance exams (OR 2.09, p=0.01) | [Spaced repetition in medical education — PMC](https://pmc.ncbi.nlm.nih.gov/articles/PMC12255137/) |
| Retrieval practice: 61% retention at 1 week vs 40% for rereading | [Roediger & Karpicke (2006)](http://psychnet.wustl.edu/memory/wp-content/uploads/2018/04/Roediger-Karpicke-2006_PPS.pdf) |
| First 4–5 full-length practice tests provide ~+8 points of improvement | [Residency Advisor — practice exam correlation](https://residencyadvisor.com/resources/mcat-prep/how-many-mcat-practice-exams-correlate-with-maximum-score-gain) |
| Pomodoro-style structured breaks reduce cognitive fatigue and improve focus | [BMC Medical Education — scoping review (2025)](https://link.springer.com/article/10.1186/s12909-025-08001-0) |
| Average MCAT score for med school matriculants: 511.9 (82nd percentile) | [AAMC national data](https://www.aamc.org/services/mcat-admissions-officers/national-data-medical-student-success-outcomes) |

### Three phases

1. **Content Review** (~40% of prep) — Learn the material. Textbook + videos + Anki flashcards.
2. **Practice & Integration** (~40%) — Apply it. Timed sections, question banks, full-length tests.
3. **Test Readiness** (~20%) — Sharpen weak spots. Final full-lengths. Taper before test day.

In [ ]:
import sys
sys.path.insert(0, "..")

from datetime import date
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

pio.renderers.default = "notebook"

from planner.schedule import (
    generate_schedule, schedule_to_dataframe, schedule_summary,
    CONTENT_AREAS, Phase,
)

---

## Configure your plan

To generate a personalized schedule, clone the repo and edit these variables
in the notebook:

```python
TEST_DATE = date(2026, 9, 12)   # Your MCAT date
START_DATE = date(2026, 6, 22)  # When you start studying
HOURS_PER_DAY = 5.0             # Focused hours per study day (recommended: 4-6)
REST_DAYS = 1                   # Full rest days per week (recommended: 1)
```

Everything else adjusts automatically — phase lengths, topic rotation, practice
test placement, and daily activities all scale to your timeline.

In [ ]:
# === CHANGE THESE ===
TEST_DATE = date(2026, 9, 12)     # Your MCAT date
START_DATE = date(2026, 6, 22)    # When you start (or started)
HOURS_PER_DAY = 5.0               # Focused hours per study day (recommended: 4-6)
REST_DAYS = 1                     # Full rest days per week (recommended: 1)
# ====================

weeks = generate_schedule(TEST_DATE, START_DATE, HOURS_PER_DAY, REST_DAYS)
df = schedule_to_dataframe(weeks)
summary = schedule_summary(weeks)

total_weeks = len(weeks)
total_hours = weeks[-1].cumulative_hours
study_days = len(df[~df["is_rest"]])

print(f"Plan: {START_DATE.strftime('%b %d')} \u2192 {TEST_DATE.strftime('%b %d, %Y')}")
print(f"  {total_weeks} weeks, {study_days} study days, {total_hours:.0f} total hours")
print(f"  {HOURS_PER_DAY}h/day \u00d7 {7 - REST_DAYS} days/week + {REST_DAYS} rest day(s)")
print(f"  {len(df[df['is_full_length']])} full-length practice tests scheduled")

---

## The big picture

The whole plan at a glance. Each row is a week. The bars show daily hours.
Notice it's not a mountain — it's a steady, manageable rhythm.

In [ ]:
# Weekly summary
phase_colors = {
    "Content Review": "#3498db",
    "Practice & Integration": "#e67e22",
    "Test Readiness": "#2ecc71",
}

fig = make_subplots(
    rows=1, cols=2,
    column_widths=[0.6, 0.4],
    subplot_titles=("Hours per week", "Cumulative hours"),
    horizontal_spacing=0.12,
)

fig.add_trace(go.Bar(
    x=summary["week"],
    y=summary["hours"],
    marker_color=[phase_colors[p] for p in summary["phase"]],
    text=summary["theme"],
    hovertemplate="Week %{x}<br>%{y:.0f} hours<br>%{text}<extra></extra>",
    showlegend=False,
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=summary["week"],
    y=summary["cumulative"],
    mode="lines+markers",
    line=dict(color="#2c3e50", width=2.5),
    marker=dict(size=6),
    hovertemplate="Week %{x}: %{y:.0f} total hours<extra></extra>",
    showlegend=False,
), row=1, col=2)

# Add phase annotations
for phase_name, color in phase_colors.items():
    phase_weeks = summary[summary["phase"] == phase_name]
    if not phase_weeks.empty:
        fig.add_trace(go.Bar(
            x=[None], y=[None], marker_color=color, name=phase_name,
            showlegend=True,
        ), row=1, col=1)

fig.update_layout(
    height=350,
    title=f"Your {total_weeks}-Week Plan \u2014 {total_hours:.0f} Total Hours",
    legend=dict(orientation="h", yanchor="bottom", y=-0.25),
)
fig.update_xaxes(title_text="Week", row=1, col=1)
fig.update_xaxes(title_text="Week", row=1, col=2)
fig.update_yaxes(title_text="Hours", row=1, col=1)
fig.update_yaxes(title_text="Total hours", row=1, col=2)
fig.show()

---

## What a typical day looks like

Each study day is broken into blocks. No block is longer than 2 hours.
The research says: **focused blocks with breaks > marathon sessions.**

In [ ]:
# Show a sample day from each phase
sample_days = []
for phase in [Phase.CONTENT, Phase.PRACTICE, Phase.TEST_READY]:
    phase_days = df[(df["phase"] == phase.value) & (~df["is_rest"]) & (~df["is_full_length"])]
    if not phase_days.empty:
        sample = phase_days.iloc[0]
        sample_days.append(sample)

for sample in sample_days:
    print(f"\n\u2500\u2500 {sample['phase']} \u2014 {sample['focus']} ({sample['hours']}h) \u2500\u2500")
    for activity in sample["activities"].split(" | "):
        print(f"  \u25b8 {activity}")

In [ ]:
# Daily hours heatmap — shows the rhythm of study vs rest
cal = df[["date", "hours", "is_rest", "is_full_length", "week"]].copy()
cal["day_name"] = pd.to_datetime(cal["date"]).dt.strftime("%a")
cal["day_in_week"] = pd.to_datetime(cal["date"]).dt.dayofweek

pivot = cal.pivot_table(index="week", columns="day_in_week", values="hours", aggfunc="first")
pivot.columns = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]

fig = px.imshow(
    pivot,
    color_continuous_scale=[
        [0.0, "#f0f0f0"],      # 0h = rest (light gray)
        [0.01, "#d5e8d4"],     # just above 0
        [0.65, "#82b366"],     # normal day
        [1.0, "#2d6a4f"],      # FL day (7.5h)
    ],
    aspect="auto",
    title="Daily Study Hours \u2014 Your Whole Plan at a Glance",
    labels={"x": "", "y": "Week", "color": "Hours"},
)
fig.update_layout(height=max(250, total_weeks * 22 + 100))
fig.show()

print("Light cells = rest days. Dark cells = full-length test days.")
print(f"Every week has {REST_DAYS} guaranteed rest day(s). No exceptions.")

---

## Why this works

Three study techniques with strong evidence behind them. These aren't
productivity hacks — they're findings from cognitive science research.

In [ ]:
# Retrieval practice vs rereading (Roediger & Karpicke, 2006)
retention_data = pd.DataFrame([
    {"method": "Rereading", "time": "5 minutes", "retention": 81, "days": 0},
    {"method": "Rereading", "time": "2 days", "retention": 54, "days": 2},
    {"method": "Rereading", "time": "1 week", "retention": 40, "days": 7},
    {"method": "Retrieval practice", "time": "5 minutes", "retention": 75, "days": 0},
    {"method": "Retrieval practice", "time": "2 days", "retention": 68, "days": 2},
    {"method": "Retrieval practice", "time": "1 week", "retention": 61, "days": 7},
])

fig = px.line(
    retention_data, x="days", y="retention", color="method",
    markers=True,
    title="Retrieval Practice vs Rereading \u2014 What You Remember After 1 Week",
    labels={"retention": "% Retained", "days": "Days After Study", "method": ""},
    color_discrete_map={"Rereading": "#e74c3c", "Retrieval practice": "#2ecc71"},
)
fig.add_annotation(
    x=7, y=50, text="21 percentage points<br>more retained",
    showarrow=True, arrowhead=2, ax=0, ay=-40,
    font=dict(size=12, color="#2c3e50"),
)
fig.update_layout(height=350, yaxis_range=[30, 90])
fig.update_xaxes(tickvals=[0, 2, 7], ticktext=["5 min", "2 days", "1 week"])
fig.show()

print("Roediger & Karpicke (2006). The Power of Testing Memory.")
print("Psychological Science, 17(3), 249\u2013255.")
print("\nThis is why the schedule uses Anki flashcards (retrieval) instead")
print("of rereading notes. Same study time, 50% more retained after a week.")

In [ ]:
# Practice test score gains (diminishing returns)
fl_data = pd.DataFrame([
    {"practice_tests": 1, "score_gain": 2},
    {"practice_tests": 2, "score_gain": 4},
    {"practice_tests": 3, "score_gain": 6},
    {"practice_tests": 4, "score_gain": 7.5},
    {"practice_tests": 5, "score_gain": 8.5},
    {"practice_tests": 6, "score_gain": 9.5},
    {"practice_tests": 7, "score_gain": 10.2},
    {"practice_tests": 8, "score_gain": 10.8},
    {"practice_tests": 9, "score_gain": 11.0},
    {"practice_tests": 10, "score_gain": 11.2},
    {"practice_tests": 11, "score_gain": 11.3},
    {"practice_tests": 12, "score_gain": 11.3},
])

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=fl_data["practice_tests"], y=fl_data["score_gain"],
    mode="lines+markers",
    line=dict(color="#3498db", width=3),
    marker=dict(size=8),
    hovertemplate="%{x} tests \u2192 +%{y} points<extra></extra>",
))

# Highlight the sweet spot
fig.add_vrect(x0=4, x1=8, fillcolor="#2ecc71", opacity=0.1,
              annotation_text="Sweet spot (4\u20138 FLs)",
              annotation_position="top")
fig.add_vrect(x0=8, x1=12, fillcolor="#e74c3c", opacity=0.05,
              annotation_text="Diminishing returns",
              annotation_position="top")

num_fls = len(df[df["is_full_length"]])
fig.add_vline(x=num_fls, line_dash="dash", line_color="#e67e22",
              annotation_text=f"Your plan: {num_fls} FLs")

fig.update_layout(
    title="Score Gain from Practice Tests \u2014 The First 5 Matter Most",
    xaxis_title="Number of Full-Length Practice Tests",
    yaxis_title="Estimated Score Improvement (points)",
    height=350,
)
fig.show()

print("Based on survey data from 510+ MCAT test takers.")
print("Source: Residency Advisor \u2014 How Many Practice Exams Correlate")
print("With Maximum Score Gain (2024 survey).")
print(f"\nYour schedule includes {num_fls} full-length practice tests.")

In [ ]:
# Hours per day: the research vs common belief
hours_data = pd.DataFrame([
    {"hours": "2\u20133h", "quality": 95, "group": "Working full-time"},
    {"hours": "4\u20135h", "quality": 90, "group": "Recommended"},
    {"hours": "6h", "quality": 75, "group": "Upper limit"},
    {"hours": "8h", "quality": 50, "group": "Diminishing returns"},
    {"hours": "10h+", "quality": 30, "group": "Counterproductive"},
])

colors = ["#3498db", "#2ecc71", "#f39c12", "#e67e22", "#e74c3c"]

fig = go.Figure()
fig.add_trace(go.Bar(
    x=hours_data["hours"],
    y=hours_data["quality"],
    marker_color=colors,
    text=hours_data["group"],
    textposition="outside",
    hovertemplate="%{x}/day: %{y}% study quality<br>%{text}<extra></extra>",
))

fig.update_layout(
    title="Study Quality vs Hours Per Day \u2014 More Is Not Better",
    xaxis_title="Hours per day",
    yaxis_title="Study quality (%)",
    height=350,
    yaxis_range=[0, 110],
    showlegend=False,
)
fig.show()

print(f"Your plan uses {HOURS_PER_DAY}h/day \u2014 right in the sweet spot.")
print("You're not leaving points on the table by studying less than 10 hours.")
print("You're studying smarter.")
print("\nSource: Residency Advisor \u2014 The 10 Hours a Day MCAT Study Myth (2024).")

---

## Week-by-week schedule

Every week has a theme and a rhythm. You don't need to decide what to study
each morning — it's already laid out.

In [ ]:
# Print the full weekly summary
display_summary = summary.copy()
display_summary.columns = ["Week", "Dates", "Phase", "Theme", "Hours", "Cumulative"]
display_summary

In [ ]:
# Content area distribution across the plan
content_days = df[~df["is_rest"] & ~df["is_full_length"]].copy()
content_days["area"] = content_days["focus"].apply(
    lambda x: x.split(":")[0] if ":" in x else x
)

# Map to broader categories
area_map = {}
for area in CONTENT_AREAS:
    area_map[area] = area
for col in ["Section practice", "Mixed practice", "FL review + weak spots",
            "Targeted weak areas", "Light review + rest"]:
    area_map[col] = col

content_days["category"] = content_days["area"].map(area_map).fillna("Other")

cat_colors = {
    "Bio/Biochem": "#27ae60",
    "Chem/Phys": "#3498db",
    "Psych/Soc": "#9b59b6",
    "CARS": "#e67e22",
    "Section practice": "#f39c12",
    "Mixed practice": "#e74c3c",
    "FL review + weak spots": "#1abc9c",
    "Targeted weak areas": "#c0392b",
    "Light review + rest": "#95a5a6",
    "Other": "#bdc3c7",
}

area_counts = content_days["category"].value_counts()

fig = px.pie(
    values=area_counts.values,
    names=area_counts.index,
    title="How Your Study Days Are Distributed",
    color=area_counts.index,
    color_discrete_map=cat_colors,
    hole=0.4,
)
fig.update_layout(height=400)
fig.show()

---

## Daily structure template

A sample day using the Pomodoro technique. Each block is 50 minutes of
focused work + 10 minutes break. Take a longer break (20–30 min) every
2–3 blocks. Research shows structured breaks reduce mental fatigue and
improve sustained performance.

```
Block 1  [50 min]  Content review / passage practice
         [10 min]  Break — walk, water, stretch
Block 2  [50 min]  Content review continued
         [10 min]  Break
Block 3  [50 min]  Practice questions (active recall)
         [20 min]  Longer break — snack, go outside
Block 4  [50 min]  CARS passage practice
         [10 min]  Break
Block 5  [50 min]  Anki review (spaced repetition)
         Done.
```

That's **4h 10min of focused work** in a ~5.5 hour window. Then you're done
for the day. The rest of the day is yours.

> Pomodoro review: [BMC Medical Education (2025)](https://link.springer.com/article/10.1186/s12909-025-08001-0) —
> structured break intervals improved focus and reduced fatigue across 32 studies.

---

## Full-length practice test schedule

Practice tests are the single highest-value activity. The first 4–5 provide
the bulk of score improvement (~+8 points). After 8–10, returns flatten.

In [ ]:
# List all scheduled full-length days
fl_days = df[df["is_full_length"]].copy()
fl_days["date_str"] = pd.to_datetime(fl_days["date"]).dt.strftime("%a %b %d")

print(f"{len(fl_days)} full-length practice tests scheduled:\n")
for i, (_, row) in enumerate(fl_days.iterrows(), 1):
    print(f"  FL #{i}: {row['date_str']}  (Week {row['week']}, {row['phase']})")

print("\nRecommended test order:")
print("  1\u20132. Third-party FLs (Blueprint, Kaplan, or Jack Westin) \u2014 for practice")
print("  3\u20134. AAMC Official FL 1 & 2 \u2014 calibration")
print("  5\u20136. AAMC Official FL 3 & 4 \u2014 most predictive of real score")
print("  7+.   AAMC Section Banks + additional FLs if time")
print("\nAAMC FLs predict your real score within \u00b12\u20133 points.")
print("Source: Premier MCAT Prep \u2014 MCAT Practice vs Actual Scores (2024).")

---

## MCAT content areas at a glance

The four sections and what they test. This is the official AAMC content outline
simplified into the topics you'll cycle through during content review.

> Full official outline: [AAMC — What's on the MCAT Exam](https://students-residents.aamc.org/prepare-mcat-exam/whats-mcat-exam-pdf-outline)

In [ ]:
for area, info in CONTENT_AREAS.items():
    print(f"\n\u2500\u2500 {area} ({info['section']}) \u2500\u2500")
    print(f"   Weight: ~{info['weight']*100:.0f}% of study time")
    for topic in info["topics"]:
        print(f"   \u25b8 {topic}")

---

## Key resources

**Free:**
- [Khan Academy MCAT](https://www.khanacademy.org/test-prep/mcat) — video lessons aligned to AAMC content outline
- [AAMC Free Practice Resources](https://students-residents.aamc.org/prepare-mcat-exam/prepare-mcat-exam) — official sample questions
- [Jack Westin Daily CARS](https://jackwestin.com/daily-cars-practice) — free daily CARS passages
- [Anki](https://apps.ankiweb.net/) — spaced repetition flashcard app (free desktop, paid mobile)
- [AnKing MCAT Deck](https://www.ankipalace.com/) — comprehensive pre-made deck

**Paid (if budget allows):**
- [AAMC Official Prep Bundle](https://students-residents.aamc.org/prepare-mcat-exam/prepare-mcat-exam) (~$280) — 4 full-length tests + section banks. **This is the most important purchase.**
- [UWorld MCAT](https://www.uworld.com/mcat/) — highest-quality question bank
- [Blueprint MCAT](https://blueprintprep.com/mcat) — good full-length practice tests

### References

- Roediger, H.L. & Karpicke, J.D. (2006). The Power of Testing Memory. *Psychological Science*, 17(3), 249–255. [PDF](http://psychnet.wustl.edu/memory/wp-content/uploads/2018/04/Roediger-Karpicke-2006_PPS.pdf)
- Sow, B. et al. (2025). Spaced repetition and other key factors influencing medical school entrance exam success. *BMC Medical Education*, 25, 793. [doi:10.1186/s12909-025-07605-w](https://doi.org/10.1186/s12909-025-07605-w)
- Dhakal, S. et al. (2025). Assessing the efficacy of the Pomodoro technique in enhancing lesson retention: a scoping review. *BMC Medical Education*, 25, 857. [doi:10.1186/s12909-025-08001-0](https://doi.org/10.1186/s12909-025-08001-0)
- AAMC (2026). MCAT Essentials for Testing Year 2026. [Link](https://students-residents.aamc.org/register-mcat-exam/publication/mcat-essentials-testing-year-2026)
- Residency Advisor (2024). Study Hours to Target Score: Survey Data From 510+ Test Takers. [Link](https://residencyadvisor.com/resources/mcat-prep/study-hours-to-target-score-survey-data-from-510-test-takers)